In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
tf.__version__

'2.17.1'

In [4]:
# PREPROCESSING THE TRAINING SET

# We are gonna apply TRANSFORMATIONS to avoid overfitting. This include rotations, flipping, zoom in, zoom out etc.
# This is known as IMAGE AUGMENTATION
train_datagen = ImageDataGenerator(
                  rescale = 1./255, # Feature scaling is applied to each pixel by divinding by 255 to get in range of 0 to 1
                  shear_range = 0.2,
                  zoom_range = 0.2,
                  horizontal_flip = True
)

# Now joining Training set
training_set = train_datagen.flow_from_directory(
                    'dataset/training_set',
                    target_size = (64,64), # size of images of training set that will be passed onto convolutional layers
                    batch_size = 32, # classic default value
                    class_mode = 'binary' # cat or dog?

)


FileNotFoundError: [Errno 2] No such file or directory: 'dataset/training_set'

In [5]:
# PREPROCESSING TEST SET IMAGES

test_datagen = ImageDataGenerator(
                  rescale = 1./255, # We wont apply other transformations in order to prevent information leakage
)

# Now joining Test set
test_set = test_datagen.flow_from_directory(
                    'dataset/test_set',
                    target_size = (64,64), # size of images of test set that will be passed onto convolutional layers
                    batch_size = 32, # classic default value
                    class_mode = 'binary' # cat or dog?

)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/test_set'

In [6]:
# INITIALIZING CNN
cnn = tf.keras.models.Sequential()

In [ ]:
# CONVOLUTIONAL LAYER

# filters are feature detectors = 32 is classic value
# kernel size = 3 for 3x3 dimension
# input shape is 64,64,3 becuase we are using coloured images, in case of b/w 64,64,1 is used. 64 is the target size we have used
# we get our feature map with this

cnn.add(tf.keras.layers.Conv2D(filters = 32, kernel_size = 3,activation ='relu',input_shape = [64,64,3]))

In [7]:
# POOLING LAYER

# we reduce the size of the convuled feature map and it is then knowns as pooled feature map
# A frame of 4 pixels is selected and the max value out of those 4 values is selected
# Thats how MaxPooling works: the max values of the frames constitues the pooled feature map
# Pool size = height of that 1 frame(no. of rows in the frame)
# Strides = by how many pixels(columns) we have shifted the frame
# Padding = when we reach the last column, there are only 2 pixels instead of 4 to select and the space for other 2 pixels is empty
# Padding = valid if we ignore those empty spaces and takes max of those 2 given pixels
# Padding = Same if we create 2 fake pixels of 0 value to fill that empty space and then consider the max value out of 4 pixels

cnn.add(tf.keras.layers.MaxPool2D(pool_size = 2, strides = 2))

In [7]:
# ADDING SECOND CONVOLUTIONAL LAYER

# We only use input shape parameter in the first layer to connect the input layer
cnn.add(tf.keras.layers.Conv2D(filters = 32, kernel_size = 3,activation ='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size = 2, strides = 2))

In [ ]:
# FLATTENING

# This will flatten the results of all these convolutions and pooling into 1D vector
# which will become input of fully connected layer
cnn.add(tf.keras.layers.Flatten())

In [ ]:
# FULLY CONNECTED LAYER

cnn.add(tf.keras.layers.Dense(units = 128, activation = 'relu'))

In [ ]:
# OUTPUT LAYER

cnn.add(tf.keras.layers.Dense(units = 1, activation = 'sigmoid'))

In [ ]:
# COMPILING CNN

cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy' , metrics = ['accuracy'])

In [8]:
# TRAINING CNN

cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

In [9]:
# PREDICTION

import numpy as np
from keras.preprocessing import image
test_image = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64,64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dms(test_image, axis = 0)
result = cnn.predict(test_image)
training_set.class_indices
if result[0][0] == 1:
  prediction = 'dog'
else:
  prediction = 'cat'


In [ ]:
print(prediction)